# Unsupervised ALBERT MLM Pre-training

This notebook demonstrates unsupervised pre-training of the ALBERT model on reaction SMILES data.

## Setup

1. **Install dependencies**: Run cell 0 to install `agave_chem`, `torch`, and `pandas`.
2. **Configure paths**: Update `TRAINING_DATA_FILE` and `SAVE_DIR` in cell 2 to point to your data and desired output directory.
3. **Tune hyperparameters**: Adjust `NUM_EPOCHS`, `BATCH_SIZE`, `WARMUP_STEPS`, etc. in cell 2.
4. **Resume from checkpoint**: Set `RESUME_FROM_CHECKPOINT` to a `.pt` checkpoint path to resume training.
5. **Early stopping**: Set `EARLY_STOPPING_PATIENCE` > 0 to stop training when validation loss stops improving.

## New features

- `save_best_model`: Saves the best model (by validation loss) to `{SAVE_DIR}/best_model.pt`
- `early_stopping_patience`: Stops training after N epochs without improvement (0 = disabled)
- `early_stopping_min_delta`: Minimum validation loss improvement to count as progress
- `resume_from_checkpoint`: Path to a `.pt` checkpoint to resume from
- `max_length`, `num_workers`, `prefetch_factor`, `masking_mode`: Now exposed as parameters

In [ ]:
!pip install -e .. --force-reinstall --no-cache-dir
!python -m pip install -U --no-cache-dir torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -U typing_extensions pydantic pydantic-core pandas

In [ ]:
import sys
from pathlib import Path

WORKFLOWS_DIR = Path.cwd().parent
if str(WORKFLOWS_DIR) not in sys.path:
    sys.path.insert(0, str(WORKFLOWS_DIR))

from loguru import logger

logger.remove()
logger.add(sys.stderr, level="ERROR")

from agave_chem.utils.chem_utils import canonicalize_reaction_smiles
from model_training_scripts.albert_mapper_unuspervised_training import (
    TrainingConfig,
    main,
)

In [ ]:
import os
import random

from rdkit import RDLogger

NUM_EPOCHS = 20
BATCH_SIZE = 64
WARMUP_STEPS = 10000
LOGGING_STEPS = 100
TRAIN_PCT = 0.99
MAX_LENGTH = 384
NUM_WORKERS = 8
PREFETCH_FACTOR = 4
MASKING_MODE = "span"
SEED = 42
SAVE_BEST_MODEL = True
EARLY_STOPPING_PATIENCE = 0
EARLY_STOPPING_MIN_DELTA = 0.0
RESUME_FROM_CHECKPOINT = None
TRAINING_DATA_FILE = "/workspace/data/uspto_all_reactions_training.txt"
SAVE_DIR = "/workspace/saved_models/albert-04-01-2026"
os.makedirs(SAVE_DIR, exist_ok=True)

RDLogger.DisableLog("rdApp.*")

In [ ]:
rxns = []
with open(TRAINING_DATA_FILE, "r") as handle:
    for i, line in enumerate(handle):
        if i % 10000 == 0:
            print(i)
        try:
            canonicalized_smiles = canonicalize_reaction_smiles(line.strip().replace("~", "."))
            if canonicalized_smiles:
                rxns.append(canonicalized_smiles)
            else:
                print(f"Cannot canonicalize {i}")
        except Exception:
            print(f"Cannot canonicalize {i}")

rxns = list(set(rxns))
random.seed(42)
random.shuffle(rxns)
rxns_train = rxns[: int(len(rxns) * 0.99)]
rxns_val = rxns[int(len(rxns) * 0.99) :]

In [ ]:
training_config = TrainingConfig(
    output_dir=SAVE_DIR,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    warmup_steps=WARMUP_STEPS,
    logging_steps=LOGGING_STEPS,
    seed=SEED,
    save_best_model=SAVE_BEST_MODEL,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
)

main(
    train_texts=rxns_train,
    val_texts=rxns_val,
    training_config=training_config,
    max_length=MAX_LENGTH,
    num_workers=NUM_WORKERS,
    prefetch_factor=PREFETCH_FACTOR,
    masking_mode=MASKING_MODE,
    resume_from_checkpoint=RESUME_FROM_CHECKPOINT,
)